# Programowanie funkcyjne w Pythonie — przykłady

Python jest językiem wieloparadygmatowym, ale wspiera wiele technik znanych z programowania funkcyjnego.

## Zakres
- funkcje jako wartości,
- funkcje wyższego rzędu,
- `lambda`,
- `map`, `filter`, `reduce`,
- comprehensions,
- czyste funkcje,
- niemutowalność,
- closures,
- `functools.partial`,
- kompozycja funkcji,
- dekoratory,
- rekurencja,
- iteratory i generatory,
- `itertools`,
- pipeline'y danych,
- proste odpowiedniki `Maybe` i `Either`,
- ćwiczenia z rozwiązaniami.

## 1. Funkcje jako wartości

In [1]:
# Funkcja jest obiektem: możemy przypisać ją do innej zmiennej.
# Zmienna f wskazuje na ten sam obiekt funkcji co square.

def square(x):
    return x * x

f = square
f(5)

25

Funkcje w Pythonie są obiektami pierwszej klasy. Można je przypisywać do zmiennych, przekazywać do innych funkcji i zwracać jako wynik.

## 2. Funkcje wyższego rzędu

In [2]:
# Funkcja apply_twice przyjmuje inną funkcję jako argument.
# func(func(value)) oznacza dwukrotne zastosowanie przekazanej funkcji.

def apply_twice(func, value):
    return func(func(value))

def increment(x):
    return x + 1

apply_twice(increment, 10)

12

## 3. `lambda`

In [3]:
# lambda tworzy krótką funkcję anonimową bez użycia def.

double = lambda x: x * 2

double(10)

20

In [4]:
# Funkcję lambda można utworzyć i wywołać od razu.

(lambda x, y: x + y)(3, 4)

7

## 4. `map`

In [5]:
# map stosuje tę samą funkcję do każdego elementu kolekcji.
# W Pythonie 3 map zwraca iterator, dlatego wynik zamieniamy na listę.

numbers = [1, 2, 3, 4, 5]

list(map(lambda x: x * 2, numbers))

[2, 4, 6, 8, 10]

In [6]:
# map nie wymaga lambdy — możemy przekazać zwykłą, nazwaną funkcję.

def square(x):
    return x * x

list(map(square, numbers))

[1, 4, 9, 16, 25]

## 5. `filter`

In [7]:
# filter pozostawia tylko te elementy, dla których predykat zwraca True.
# Tutaj wybieramy wyłącznie liczby parzyste.

numbers = range(1, 21)

list(filter(lambda x: x % 2 == 0, numbers))

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]

## 6. `reduce`

In [8]:
# reduce scala całą sekwencję do jednej wartości.
# acc jest akumulatorem, a x kolejnym elementem listy.

from functools import reduce

numbers = [1, 2, 3, 4, 5]

reduce(lambda acc, x: acc + x, numbers, 0)

15

In [9]:
# Zmieniamy operację z dodawania na mnożenie; wartość początkowa to 1.

reduce(lambda acc, x: acc * x, numbers, 1)

120

`reduce` odpowiada idei fold znanej z języków funkcyjnych.

## 7. List comprehensions

In [10]:
# List comprehension tworzy nową listę na podstawie transformacji elementów.

[x * x for x in range(1, 11)]

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

In [11]:
# Warunek po if pełni rolę filtra.

[x for x in range(1, 31) if x % 3 == 0]

[3, 6, 9, 12, 15, 18, 21, 24, 27, 30]

In [12]:
# Transformację i filtrowanie można połączyć w jednym wyrażeniu.

[x * x for x in range(1, 21) if x % 2 == 0]

[4, 16, 36, 64, 100, 144, 196, 256, 324, 400]

## 8. Czyste funkcje

In [13]:
# add jest funkcją czystą: wynik zależy wyłącznie od argumentów.
# Ponowne wywołanie z tymi samymi danymi daje ten sam rezultat.

def add(a, b):
    return a + b

print(add(2, 3))
print(add(2, 3))

5
5


Czysta funkcja dla tych samych argumentów zwraca ten sam wynik i nie modyfikuje stanu poza sobą.

## 9. Efekt uboczny

In [14]:
# Ta funkcja modyfikuje zmienną globalną, więc ma efekt uboczny.
# Wynik zależy od historii wcześniejszych wywołań.

counter = 0

def impure_increment():
    global counter
    counter += 1
    return counter

print(impure_increment())
print(impure_increment())

1
2


## 10. Niemutowalność

In [15]:
# Tuple jest niemutowalna — zamiast zmieniać numbers tworzymy nową krotkę.
# Taki styl ogranicza niejawne zmiany stanu.

numbers = (1, 2, 3, 4, 5)

new_numbers = tuple(x * 2 for x in numbers)

print(numbers)
print(new_numbers)

(1, 2, 3, 4, 5)
(2, 4, 6, 8, 10)


## 11. Immutable `dataclass`

In [16]:
# frozen=True blokuje zmianę pól po utworzeniu obiektu.
# Point zachowuje się dzięki temu jak niemutowalna wartość.

from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
p

Point(x=1.0, y=2.0)

## 12. Closure

In [17]:
# Funkcja wewnętrzna pamięta wartość factor z otaczającego zakresu.
# To właśnie mechanizm closure (domknięcia).

def multiplier(factor):
    def multiply(value):
        return value * factor
    return multiply

times2 = multiplier(2)
times10 = multiplier(10)

print(times2(5))
print(times10(5))

10
50


## 13. Funkcja zwracająca funkcję

In [18]:
# power nie zwraca liczby, lecz nową funkcję.
# Każda utworzona funkcja pamięta własną wartość exponent.

def power(exponent):
    return lambda x: x ** exponent

square = power(2)
cube = power(3)

print(square(5))
print(cube(5))

25
125


## 14. `functools.partial`

In [19]:
# partial tworzy nową funkcję przez ustalenie części argumentów.
# double i triple nadal oczekują jednego brakującego argumentu.

from functools import partial

def multiply(a, b):
    return a * b

double = partial(multiply, 2)
triple = partial(multiply, 3)

print(double(10))
print(triple(10))

20
30


## 15. Currying — prosta implementacja

In [20]:
# Currying rozbija funkcję wieloargumentową na serię funkcji jednoargumentowych.
# curry_add(10) zwraca funkcję, która pamięta wartość 10.

def curry_add(a):
    def inner(b):
        return a + b
    return inner

add10 = curry_add(10)

add10(5)

15

## 16. Kompozycja funkcji

In [21]:
# compose(f, g)(x) wykonuje najpierw g(x), a potem f(...).
# To prosty odpowiednik operatora kompozycji (.) z Haskella.

def compose(f, g):
    return lambda x: f(g(x))

def double(x):
    return x * 2

def increment(x):
    return x + 1

double_after_increment = compose(double, increment)

double_after_increment(10)

22

## 17. Kompozycja wielu funkcji

In [22]:
# reduce buduje jedną funkcję z wielu funkcji składowych.
# W tym wariancie funkcje wykonują się od prawej do lewej.

from functools import reduce

def compose_many(*functions):
    return reduce(
        lambda f, g: lambda x: f(g(x)),
        functions
    )

pipeline = compose_many(
    lambda x: x + 100,
    lambda x: x * 2,
    lambda x: x - 3
)

pipeline(10)

114

## 18. Pipeline danych

In [23]:
# Generator expression filtruje liczby parzyste i od razu podnosi je do kwadratu.
# sum konsumuje wyniki bez potrzeby tworzenia listy pośredniej.

data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = sum(
    x * x
    for x in data
    if x % 2 == 0
)

result

220

## 19. Pipeline z `map`, `filter`, `reduce`

In [24]:
# Ten sam pipeline zapisany jawnie jako filter -> map -> reduce.
# Dane przepływają od najbardziej zagnieżdżonej operacji na zewnątrz.

from functools import reduce

result = reduce(
    lambda acc, x: acc + x,
    map(
        lambda x: x * x,
        filter(lambda x: x % 2 == 0, data)
    ),
    0
)

result

220

## 20. Funkcje w kolekcjach

In [25]:
# Funkcje można przechowywać w kolekcjach tak samo jak inne obiekty.
# Pętla kolejno przekazuje wynik poprzedniej operacji do następnej.

operations = [
    lambda x: x + 1,
    lambda x: x * 2,
    lambda x: x ** 2
]

value = 3

for operation in operations:
    value = operation(value)

value

64

## 21. Słownik strategii

In [26]:
# Słownik może mapować nazwy strategii na funkcje.
# Pozwala to wybierać zachowanie dynamicznie bez rozbudowanych if/elif.

strategies = {
    "double": lambda x: x * 2,
    "square": lambda x: x ** 2,
    "negate": lambda x: -x
}

strategies["square"](5)

25

## 22. Moduł `operator`

In [27]:
# Moduł operator dostarcza funkcje odpowiadające operatorom języka.
# Są wygodne tam, gdzie chcemy przekazać operację jako wartość.

import operator

print(operator.add(2, 3))
print(operator.mul(4, 5))

reduce(operator.add, [1, 2, 3, 4], 0)

5
20


10

## 23. Sortowanie przez funkcję

In [28]:
# Parametr key mówi sorted, jaką wartość wyliczyć dla każdego elementu.
# Tutaj sortujemy osoby według wieku, czyli elementu o indeksie 1.

people = [
    ("Anna", 35),
    ("Jan", 22),
    ("Maria", 29)
]

sorted(people, key=lambda person: person[1])

[('Jan', 22), ('Maria', 29), ('Anna', 35)]

## 24. `operator.itemgetter`

In [29]:
# itemgetter(1) jest gotową funkcją pobierającą element o indeksie 1.
# Może zastąpić prostą lambdę i często jest bardziej czytelny.

from operator import itemgetter

sorted(people, key=itemgetter(1))

[('Jan', 22), ('Maria', 29), ('Anna', 35)]

## 25. Rekurencja

In [30]:
# Rekurencja wymaga warunku bazowego, tutaj n == 0.
# Dla pozostałych wartości funkcja wywołuje samą siebie dla mniejszego n.

def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)

factorial(5)

120

Python nie wykonuje optymalizacji tail recursion, dlatego rekurencja nie jest tak centralnym narzędziem jak w Haskellu.

## 26. Iteratory

In [31]:
# iter tworzy iterator, a next pobiera kolejny element.
# Iterator przechowuje informację o aktualnej pozycji w strumieniu.

numbers = iter([1, 2, 3])

print(next(numbers))
print(next(numbers))
print(next(numbers))

1
2
3


## 27. Generatory

In [32]:
# yield zwraca jedną wartość i zawiesza wykonanie funkcji.
# Przy kolejnym żądaniu generator kontynuuje od miejsca zatrzymania.

def squares(n):
    for x in range(n):
        yield x * x

list(squares(10))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

## 28. Generator expression

In [33]:
# Generator expression jest leniwym odpowiednikiem list comprehension.
# Nie tworzy miliona elementów w pamięci — oblicza je na żądanie.

generator = (x * x for x in range(1_000_000))

[next(generator) for _ in range(5)]

[0, 1, 4, 9, 16]

## 29. Nieskończony generator

In [34]:
# Pętla while True może reprezentować nieskończony strumień.
# Dzięki yield pobieramy tylko tyle wartości, ile rzeczywiście potrzebujemy.

def naturals():
    n = 1
    while True:
        yield n
        n += 1

g = naturals()

[next(g) for _ in range(10)]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

## 30. `itertools`

In [35]:
# itertools dostarcza gotowe narzędzia do leniwego przetwarzania iteratorów.
# islice ogranicza nawet nieskończony iterator do wybranego fragmentu.

from itertools import count, islice, cycle, repeat, chain, takewhile

print(list(islice(count(1), 10)))
print(list(islice(cycle(["A", "B", "C"]), 8)))
print(list(repeat("Python", 4)))
print(list(chain([1, 2], [3, 4], [5])))
print(list(takewhile(lambda x: x < 10, [1, 3, 5, 9, 12, 2])))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
['A', 'B', 'C', 'A', 'B', 'C', 'A', 'B']
['Python', 'Python', 'Python', 'Python']
[1, 2, 3, 4, 5]
[1, 3, 5, 9]


## 31. `zip`

In [36]:
# zip łączy elementy z kilku sekwencji według ich pozycji.
# Wynikiem jest iterator krotek (name, age).

names = ["Anna", "Jan", "Maria"]
ages = [30, 25, 40]

list(zip(names, ages))

[('Anna', 30), ('Jan', 25), ('Maria', 40)]

## 32. Dekorator jako funkcja wyższego rzędu

In [37]:
# Dekorator przyjmuje funkcję i zwraca nową funkcję wrapper.
# wrapper dodaje zachowanie przed i po właściwym wywołaniu bez zmiany add.

def debug(func):
    def wrapper(*args, **kwargs):
        print(f"Wywołuję {func.__name__} z {args=} {kwargs=}")
        result = func(*args, **kwargs)
        print(f"Wynik: {result}")
        return result
    return wrapper

@debug
def add(a, b):
    return a + b

add(2, 3)

Wywołuję add z args=(2, 3) kwargs={}
Wynik: 5


5

## 33. Memoization

In [38]:
# lru_cache zapamiętuje wyniki wcześniejszych wywołań.
# Jest szczególnie skuteczny dla czystych funkcji rekurencyjnych, takich jak Fibonacci.

from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

[fib(n) for n in range(20)]

[0,
 1,
 1,
 2,
 3,
 5,
 8,
 13,
 21,
 34,
 55,
 89,
 144,
 233,
 377,
 610,
 987,
 1597,
 2584,
 4181]

## 34. Uproszczone `Maybe` przez `None`

In [39]:
# None reprezentuje brak poprawnego wyniku.
# To najprostszy pythonowy odpowiednik idei Maybe/Nothing.

def safe_divide(a, b):
    if b == 0:
        return None
    return a / b

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
None


## 35. Własny typ `Maybe`

In [40]:
# Generic pozwala opisać Maybe[T] dla różnych typów wartości.
# map zmienia wartość tylko wtedy, gdy nie jest ona None.

from dataclasses import dataclass
from typing import Generic, TypeVar, Callable

T = TypeVar("T")
U = TypeVar("U")

@dataclass(frozen=True)
class Maybe(Generic[T]):
    value: T | None

    def map(self, func: Callable[[T], U]) -> "Maybe[U]":
        if self.value is None:
            return Maybe(None)
        return Maybe(func(self.value))

print(Maybe(10).map(lambda x: x * 2))
print(Maybe(None).map(lambda x: x * 2))

Maybe(value=20)
Maybe(value=None)


## 36. Prosty odpowiednik `Either`

In [41]:
# Left reprezentuje błąd, a Right poprawny wynik.
# Dzięki temu funkcja zwraca błąd jako wartość zamiast od razu rzucać wyjątek.

from dataclasses import dataclass

@dataclass(frozen=True)
class Left:
    error: str

@dataclass(frozen=True)
class Right:
    value: object

def safe_divide_either(a, b):
    if b == 0:
        return Left("Dzielenie przez zero")
    return Right(a / b)

print(safe_divide_either(10, 2))
print(safe_divide_either(10, 0))

Right(value=5.0)
Left(error='Dzielenie przez zero')


## 37. Pattern matching

In [42]:
# match rozpoznaje wariant wyniku na podstawie struktury obiektu.
# Przypomina pattern matching znany z języków funkcyjnych.

def describe_result(result):
    match result:
        case Right(value):
            return f"Wynik: {value}"
        case Left(error):
            return f"Błąd: {error}"

print(describe_result(safe_divide_either(10, 2)))
print(describe_result(safe_divide_either(10, 0)))

Wynik: 5.0
Błąd: Dzielenie przez zero


## 38. Funkcyjna transformacja danych

In [43]:
# Tworzymy nowe słowniki zamiast modyfikować istniejące rekordy users.
# {**user, 'adult': True} kopiuje pola i dodaje nowe pole.

users = [
    {"name": "Anna", "age": 30},
    {"name": "Jan", "age": 17},
    {"name": "Maria", "age": 42}
]

adults = [
    {**user, "adult": True}
    for user in users
    if user["age"] >= 18
]

adults

[{'name': 'Anna', 'age': 30, 'adult': True},
 {'name': 'Maria', 'age': 42, 'adult': True}]

## 39. Pipeline transakcji

In [44]:
# Pipeline deklaruje transformację danych w jednym wyrażeniu.
# Najpierw filtrujemy transakcje typu income, potem pobieramy amount i sumujemy.

transactions = [
    {"amount": 120, "type": "income"},
    {"amount": 50, "type": "expense"},
    {"amount": 200, "type": "income"},
    {"amount": 25, "type": "expense"},
]

income_total = sum(
    transaction["amount"]
    for transaction in transactions
    if transaction["type"] == "income"
)

income_total

320

## 40. Styl imperatywny vs deklaratywny

In [45]:
# Wersja imperatywna jawnie zarządza pustą listą, pętlą i append.
# Stan result zmienia się w trakcie działania programu.

numbers = [1, 2, 3, 4, 5, 6]

result = []
for x in numbers:
    if x % 2 == 0:
        result.append(x * x)

result

[4, 16, 36]

In [46]:
# Wersja deklaratywna opisuje wynik: kwadraty liczb parzystych.
# Nie zarządzamy ręcznie kolejnymi zmianami listy wynikowej.

[x * x for x in numbers if x % 2 == 0]

[4, 16, 36]

In [47]:
# rozwiązanie tutaj

In [56]:
# rozwiązanie tutaj

## 41. Python vs Haskell

| Koncept | Haskell | Python |
|---|---|---|
| map | `map` | `map` |
| filter | `filter` | `filter` |
| fold | `foldl`, `foldr` | `functools.reduce` |
| lambda | `\x -> ...` | `lambda x: ...` |
| kompozycja | `(.)` | własna funkcja |
| partial application | naturalna | `functools.partial` |
| leniwe sekwencje | naturalne | generators / iterators |
| Maybe | `Maybe` | `None` / własny typ |
| Either | `Either` | wyjątki / własny typ |
| pattern matching | podstawowy mechanizm | `match` |
| niemutowalność | naturalna | opcjonalna |
